# Intermediate 02 — MCP Gateway Security

Authorize MCP calls independently of discovery and model output. The lab proves server trust, identity binding, token audience, scope, exact schemas, quotas, and non-passthrough.

## 1. Load the course lab

The notebook imports the reusable course module rather than copying its security logic.

In [ ]:
import runpy
from datetime import datetime, timedelta, timezone
ns = runpy.run_path('02_mcp_gateway.py')
Gateway, ClientIdentity, AccessToken, ToolSpec, ToolCall = (ns[name] for name in ('Gateway', 'ClientIdentity', 'AccessToken', 'ToolSpec', 'ToolCall'))
now = datetime(2026, 9, 12, tzinfo=timezone.utc)
gateway = Gateway({'research-mcp-v1': {'search_policy': ToolSpec('search_policy', 'policy:search', frozenset({'query'}))}}, per_subject_limit=2)
identity = ClientIdentity('research-agent', 'north')
token = AccessToken('research-agent', 'north', 'mcp-gateway', frozenset({'policy:search'}), now + timedelta(minutes=5), 'opaque-7')
safe = ToolCall('research-mcp-v1', 'search_policy', {'query': 'retention'})

## 2. Establish the safe baseline

Observe the trusted inputs and the decision evidence before injecting failures.

In [ ]:
allowed = gateway.dispatch(identity, token, safe, now=now)
assert allowed['status'] == 'allow'
allowed

## 3. Inject an attack

Change one security-relevant boundary and keep the rest of the fixture stable.

In [ ]:
attacks = [
 gateway.dispatch(identity, AccessToken('research-agent','north','other-api',token.scopes,token.expires_at,'opaque-8'), safe, now=now),
 gateway.dispatch(identity, token, ToolCall('evil-mcp','search_policy',{'query':'x'}), now=now),
 gateway.dispatch(identity, token, ToolCall('research-mcp-v1','search_policy',{'query':'x','admin':'true'}), now=now),
]
[(r['status'], r['receipt'].reason) for r in attacks]

## 4. Attempt a bypass

The assertions below make the security property executable and regression-testable.

In [ ]:
assert [r['receipt'].reason for r in attacks] == ['token-audience', 'untrusted-server', 'argument-schema']
assert 'opaque-7' not in repr(allowed)
assert all(r['receipt'].decision == 'deny' for r in attacks)

## 5. Evaluate observable outcomes

Use explicit denominators or counts. Private model reasoning is neither required nor recorded.

In [ ]:
from collections import Counter
Counter(receipt.reason for receipt in gateway.receipts)

## 6. Exercise a second failure mode

In [ ]:
bypass = gateway.dispatch(identity, token, safe, now=now)
assert bypass['status'] == 'allow'
limited = gateway.dispatch(identity, token, safe, now=now)
assert limited['receipt'].reason == 'rate-limit'
limited

## 7. Production replacement

Production replacement: authenticated workload identity, OAuth resource indicators, separate upstream tokens, TLS, egress policy, durable quotas, protected receipts, revocation, and incident ownership. Tool results remain untrusted after an allowed call.

## Checkpoint

Explain which trusted component enforces the invariant, what evidence proves the decision, and what residual risk remains.